In [ ]:
# =============================================================================
# Pipeline iterativo para crear múltiples batches de datos
# =============================================================================

import os
import numpy as np
import pandas as pd

# -----------------------------------------------------------------------------
# Configuración de rutas
# -----------------------------------------------------------------------------
data_dir = "../entrega_2/airflow/data"   # AJUSTAR SEGÚN TU ESTRUCTURA DE CARPETAS
output_dir = "../entrega_3/data/"

# Crear directorio de salida si no existe
os.makedirs(output_dir, exist_ok=True)
print(f"✓ Directorio de salida: {output_dir}")

# -----------------------------------------------------------------------------
# Definición de archivos para cada batch
# -----------------------------------------------------------------------------
# Batch 0: solo transacciones.parquet
# Batch 1: transacciones.parquet + transacciones_1.parquet
# Batch 2: transacciones.parquet + transacciones_1.parquet + transacciones_2.parquet
# Batch 3: transacciones.parquet + transacciones_1.parquet + transacciones_2.parquet + transacciones_3.parquet
# Batch 4: transacciones.parquet + transacciones_1.parquet + ... + transacciones_4.parquet

batches_config = [
    {"batch_num": 0, "archivos_adicionales": []},
    {"batch_num": 1, "archivos_adicionales": ["transacciones_1.parquet"]},
    {"batch_num": 2, "archivos_adicionales": ["transacciones_1.parquet", "transacciones_2.parquet"]},
    {"batch_num": 3, "archivos_adicionales": ["transacciones_1.parquet", "transacciones_2.parquet", "transacciones_3.parquet"]},
    {"batch_num": 4, "archivos_adicionales": ["transacciones_1.parquet", "transacciones_2.parquet", "transacciones_3.parquet", "transacciones_4.parquet"]},
]

# -----------------------------------------------------------------------------
# Función para procesar un batch completo
# -----------------------------------------------------------------------------
def procesar_batch(batch_num, archivos_adicionales):
    """
    Procesa un batch: carga datos, concatena transacciones según configuración,
    aplica todo el procesamiento y retorna el df_final.
    """
    print("\n" + "=" * 70)
    print(f"PROCESANDO BATCH {batch_num}")
    print("=" * 70)
    
    # -------------------------------------------------------------------------
    # 1. CARGA DE DATOS
    # -------------------------------------------------------------------------
    print("\nPASO 1: Cargando datos crudos...")
    
    clientes = pd.read_parquet(os.path.join(data_dir, "clientes.parquet"))
    productos = pd.read_parquet(os.path.join(data_dir, "productos.parquet"))
    transacciones = pd.read_parquet(os.path.join(data_dir, "transacciones.parquet"))
    
    print(f"✓ Clientes: {len(clientes):,}")
    print(f"✓ Productos: {len(productos):,}")
    print(f"✓ Transacciones base: {len(transacciones):,}")
    
    # Concatenar archivos adicionales según batch
    transacciones_adicionales = []
    for archivo in archivos_adicionales:
        path = os.path.join(data_dir, archivo)
        if os.path.exists(path):
            df_temp = pd.read_parquet(path)
            transacciones_adicionales.append(df_temp)
            print(f"✓ {archivo}: {len(df_temp):,}")
        else:
            print(f"⚠ {archivo}: NO ENCONTRADO")
    
    if transacciones_adicionales:
        transacciones = pd.concat([transacciones] + transacciones_adicionales, ignore_index=True)
        print(f"✓ Total concatenado: {len(transacciones):,}")
    
    # -------------------------------------------------------------------------
    # 2. CASTING Y LIMPIEZA
    # -------------------------------------------------------------------------
    print("\nPASO 2: Aplicando tipos y limpieza...")
    
    clientes = clientes.astype({
        "customer_id": "string",
        "region_id": "string",
        "zone_id": "string",
        "customer_type": "category",
    })
    
    productos = productos.astype({
        "product_id": "string",
        "brand": "category",
        "category": "category",
        "sub_category": "category",
        "segment": "category",
        "package": "category",
    })
    
    transacciones = transacciones.astype({
        "customer_id": "string",
        "product_id": "string",
        "order_id": "string",
    })
    
    transacciones["purchase_date"] = pd.to_datetime(transacciones["purchase_date"], errors="coerce")
    transacciones["year"] = transacciones["purchase_date"].dt.year
    transacciones["month"] = transacciones["purchase_date"].dt.month
    transacciones["day"] = transacciones["purchase_date"].dt.day
    
    print("✓ Tipos aplicados")
    
    # -------------------------------------------------------------------------
    # 3. DEDUPLICACIÓN
    # -------------------------------------------------------------------------
    print("\nPASO 3: Deduplicación...")
    
    registros_iniciales = len(transacciones)
    transacciones = transacciones.drop_duplicates()
    print(f"✓ Duplicados eliminados: {registros_iniciales - len(transacciones):,}")
    
    cols_clave = ["customer_id", "product_id", "order_id", "purchase_date"]
    mask_parcial = transacciones.groupby(cols_clave)["items"].transform("nunique") > 1
    df_parcial = transacciones[mask_parcial].copy()
    df_no_parcial = transacciones[~mask_parcial].copy()
    
    if len(df_parcial) > 0:
        df_parcial_fix = (
            df_parcial.groupby(cols_clave, as_index=False)
            .agg(
                items=("items", "sum"),
                year=("year", "first"),
                month=("month", "first"),
                day=("day", "first"),
            )
        )
        transacciones = pd.concat([df_no_parcial, df_parcial_fix], ignore_index=True)
        transacciones = transacciones.sort_values(cols_clave).reset_index(drop=True)
        print(f"✓ Parciales consolidados: {len(df_parcial):,} → {len(df_parcial_fix):,}")
    
    antes = len(transacciones)
    transacciones = transacciones[transacciones["items"] > 0]
    print(f"✓ Items <= 0 eliminados: {antes - len(transacciones):,}")
    print(f"✓ Transacciones finales: {len(transacciones):,}")
    
    # -------------------------------------------------------------------------
    # 4. MERGE
    # -------------------------------------------------------------------------
    print("\nPASO 4: Cruce de información...")
    
    df = transacciones.merge(clientes, on="customer_id", how="left")
    df = df.merge(productos, on="product_id", how="left")
    df = df[df["items"] > 0]
    
    for col in ["customer_type", "brand", "category", "zone_id"]:
        if col in df.columns:
            df[col] = df[col].astype(str)
    
    print(f"✓ Merge completo: {len(df):,}")
    
    # -------------------------------------------------------------------------
    # 5. PANEL SEMANAL
    # -------------------------------------------------------------------------
    print("\nPASO 5: Panel semanal...")
    
    df_ = df.copy()
    df_["purchase_date"] = pd.to_datetime(df_["purchase_date"], errors="coerce")
    df_ = df_.dropna(subset=["purchase_date"])
    
    panel = (
        df_.set_index("purchase_date")
        .groupby(["customer_id", "product_id"])
        .resample("W-MON")
        .size()
        .rename("purchased_count")
        .reset_index()
    )
    
    panel.rename(columns={"purchase_date": "week_t"}, inplace=True)
    panel["compra_o_no"] = (panel["purchased_count"] > 0).astype(int)
    panel = panel.sort_values(["customer_id", "product_id", "week_t"])
    
    panel["y"] = panel.groupby(["customer_id", "product_id"])["compra_o_no"].shift(-1)
    panel = panel.dropna(subset=["y"]).copy()
    panel["y"] = panel["y"].astype(int)
    
    panel["week_t_plus_1"] = panel["week_t"] + pd.Timedelta(days=7)
    
    iso = panel["week_t"].dt.isocalendar()
    panel["semana"] = iso.year.astype(str) + "-" + iso.week.astype(str).str.zfill(2)
    
    iso1 = panel["week_t_plus_1"].dt.isocalendar()
    panel["semana_siguiente_str"] = iso1.year.astype(str) + "-" + iso1.week.astype(str).str.zfill(2)
    
    print(f"✓ Panel creado: {len(panel):,}")
    
    df_copy = panel[[
        "customer_id", "product_id", "week_t", "week_t_plus_1",
        "semana", "semana_siguiente_str", "purchased_count", "compra_o_no", "y"
    ]].reset_index(drop=True)
    
    # -------------------------------------------------------------------------
    # 6. EXPANSIÓN CARTESIANA
    # -------------------------------------------------------------------------
    print("\nPASO 6: Expansión cartesiana...")
    
    customers = df["customer_id"].drop_duplicates().sort_values().to_numpy()
    products = df["product_id"].drop_duplicates().sort_values().to_numpy()
    weeks = pd.to_datetime(df_copy["week_t"]).drop_duplicates().sort_values().to_list()
    
    print(f"✓ Clientes: {len(customers):,}, Productos: {len(products):,}, Semanas: {len(weeks):,}")
    
    frames = []
    n_prod = len(products)
    n_cust = len(customers)
    
    for i, wk in enumerate(weeks, 1):
        if i % 10 == 0 or i == len(weeks):
            print(f"  Procesando semana {i}/{len(weeks)}...", end="\r")
        
        wk_ts = pd.Timestamp(wk)
        wk_plus1 = wk_ts + pd.Timedelta(days=7)
        
        iso0 = wk_ts.isocalendar()
        iso1 = wk_plus1.isocalendar()
        semana_str = f"{iso0.year}-{int(iso0.week):02d}"
        semana_next_str = f"{iso1.year}-{int(iso1.week):02d}"
        
        base = pd.DataFrame({
            "customer_id": np.repeat(customers, n_prod),
            "product_id": np.tile(products, n_cust),
            "week_t": wk_ts,
            "week_t_plus_1": wk_plus1,
            "semana": semana_str,
            "semana_siguiente_str": semana_next_str,
        })
        
        wk_rows = df_copy.loc[
            df_copy["week_t"].eq(wk_ts),
            ["customer_id", "product_id", "purchased_count", "compra_o_no", "y"]
        ]
        
        merged = base.merge(wk_rows, on=["customer_id", "product_id"], how="left")
        
        for c in ["purchased_count", "compra_o_no", "y"]:
            merged[c] = merged[c].fillna(0).astype(int)
        
        frames.append(merged)
    
    print()
    df_full = pd.concat(frames, ignore_index=True)
    print(f"✓ Cartesiano: {len(df_full):,}")
    
    # -------------------------------------------------------------------------
    # 7. DIMENSIONES
    # -------------------------------------------------------------------------
    print("\nPASO 7: Agregando dimensiones...")
    
    cols_cli = [
        "customer_id", "customer_type", "X", "Y", "zone_id", 
        "region_id", "num_deliver_per_week", "num_visit_per_week"
    ]
    cols_cli = [c for c in cols_cli if c in df.columns]
    
    cols_prod = [
        "product_id", "brand", "category", "sub_category",
        "segment", "package", "size"
    ]
    cols_prod = [c for c in cols_prod if c in df.columns]
    
    dim_clientes = df[cols_cli].drop_duplicates(subset=["customer_id"]).reset_index(drop=True)
    dim_productos = df[cols_prod].drop_duplicates(subset=["product_id"]).reset_index(drop=True)
    
    df_final = (
        df_full
        .merge(dim_clientes, on="customer_id", how="left")
        .merge(dim_productos, on="product_id", how="left")
    )
    
    df_final["semana_num"] = (
        df_final["semana"].str.split("-").str[0].astype(int) * 100
        + df_final["semana"].str.split("-").str[1].astype(int)
    )
    
    print(f"✓ Dataset final: {len(df_final):,} registros, {len(df_final.columns)} columnas")
    
    return df_final


# -----------------------------------------------------------------------------
# LOOP PRINCIPAL: Procesar todos los batches
# -----------------------------------------------------------------------------
print("\n" + "=" * 70)
print("INICIO DEL PROCESAMIENTO DE BATCHES")
print("=" * 70)

for config in batches_config:
    batch_num = config["batch_num"]
    archivos_adicionales = config["archivos_adicionales"]
    
    # Procesar el batch
    df_batch = procesar_batch(batch_num, archivos_adicionales)
    
    # Guardar como CSV
    output_filename = f"data_batch_{batch_num:02d}.csv"
    output_path = os.path.join(output_dir, output_filename)
    
    print(f"\n💾 Guardando: {output_filename}...")
    df_batch.to_csv(output_path, index=False)
    
    file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
    print(f"✓ Archivo guardado: {output_path}")
    print(f"✓ Tamaño: {file_size_mb:.2f} MB")
    print(f"✓ Forma: {df_batch.shape}")

# -----------------------------------------------------------------------------
# RESUMEN FINAL
# -----------------------------------------------------------------------------
print("\n" + "=" * 70)
print("PROCESO COMPLETADO")
print("=" * 70)
print(f"✓ Total de batches creados: {len(batches_config)}")
print(f"✓ Archivos guardados en: {output_dir}")
print("\nArchivos generados:")
for config in batches_config:
    filename = f"data_batch_{config['batch_num']:02d}.csv"
    filepath = os.path.join(output_dir, filename)
    if os.path.exists(filepath):
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        print(f"  ✓ {filename} ({size_mb:.2f} MB)")
print("\n🎉 Pipeline completado exitosamente!")